# ensemble 

In [1]:
import pandas as pd
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
from shapely.ops import unary_union
import glob
import os

# --- Global settings ---
getcontext().prec = 50 
scale_factor = Decimal('1e18')

# --- Core class (simplified, only for score calculation) ---
class ChristmasTree:
    def __init__(self, center_x='0', center_y='0', angle='0'):
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)
        self.polygon = self._create_polygon()

    def _create_polygon(self):
        trunk_w = Decimal('0.15'); trunk_h = Decimal('0.2')
        base_w = Decimal('0.7'); base_y = Decimal('0.0')
        mid_w = Decimal('0.4'); tier_2_y = Decimal('0.25')
        top_w = Decimal('0.25'); tier_1_y = Decimal('0.5')
        tip_y = Decimal('0.8'); trunk_bottom_y = -trunk_h

        initial_polygon = Polygon([
            (Decimal('0.0') * scale_factor, tip_y * scale_factor),
            (top_w / Decimal('2') * scale_factor, tier_1_y * scale_factor),
            (top_w / Decimal('4') * scale_factor, tier_1_y * scale_factor),
            (mid_w / Decimal('2') * scale_factor, tier_2_y * scale_factor),
            (mid_w / Decimal('4') * scale_factor, tier_2_y * scale_factor),
            (base_w / Decimal('2') * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal('2') * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal('2') * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal('2')) * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal('2')) * scale_factor, base_y * scale_factor),
            (-(base_w / Decimal('2')) * scale_factor, base_y * scale_factor),
            (-(mid_w / Decimal('4')) * scale_factor, tier_2_y * scale_factor),
            (-(mid_w / Decimal('2')) * scale_factor, tier_2_y * scale_factor),
            (-(top_w / Decimal('4')) * scale_factor, tier_1_y * scale_factor),
            (-(top_w / Decimal('2')) * scale_factor, tier_1_y * scale_factor),
        ])
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        return affinity.translate(
            rotated,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor)
        )

def get_tree_list_side_length(tree_list: list[ChristmasTree]) -> Decimal:
    if not tree_list:
        return Decimal('0')
    all_polygons = [t.polygon for t in tree_list]
    bounds = unary_union(all_polygons).bounds
    width = bounds[2] - bounds[0]
    height = bounds[3] - bounds[1]
    return Decimal(max(width, height)) / scale_factor

def get_total_score(dict_of_side_length: dict[str, Decimal]):
    score = 0
    for k, v in dict_of_side_length.items():
        score += v ** 2 / Decimal(k)
    return score

def parse_csv(csv_path):
    """Load a single CSV and return DataFrames grouped by Group ID and their side lengths"""
    print(f'Loading: {csv_path}')
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Skipping {csv_path}: {e}")
        return None, None

    # Clean data
    for col in ['x', 'y', 'deg']:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip('s')

    df[['group_id', 'item_id']] = df['id'].str.split('_', n=2, expand=True)

    group_data_map = {}   # Store original DataFrame for each group (used for final output)
    group_score_map = {}  # Store computed side length for each group

    # Group by group_id in advance for efficiency
    grouped = df.groupby('group_id')

    for group_id, group_df in grouped:
        # Rebuild Tree objects to compute accurate score
        tree_list = [
            ChristmasTree(center_x=row['x'], center_y=row['y'], angle=row['deg'])
            for _, row in group_df.iterrows()
        ]

        # Compute side length score for this group
        score = get_tree_list_side_length(tree_list)

        group_data_map[group_id] = group_df
        group_score_map[group_id] = score

    return group_data_map, group_score_map

# --- Main logic ---

def ensemble_files(file_list, output_file='/kaggle/working/submission_.csv'):
    if not file_list:
        print("No files provided!")
        return

    # 1. Store best results
    best_group_data = {}    # {group_id: dataframe_rows}
    best_group_scores = {}  # {group_id: min_side_length}

    # 2. Iterate through all files
    for fpath in file_list:
        data_map, score_map = parse_csv(fpath)
        if data_map is None:
            continue

        for group_id, score in score_map.items():
            # First time seeing this group, or found a smaller side length
            if group_id not in best_group_scores:
                best_group_scores[group_id] = score
                best_group_data[group_id] = data_map[group_id]

            elif score < best_group_scores[group_id]:
                diff = best_group_scores[group_id] - score
                # Only replace when difference is significant (avoid floating error)
                if diff > 1e-18:
                    print(
                        f"  [Group {group_id}] Improved! "
                        f"{best_group_scores[group_id]:.6f} -> {score:.6f} "
                        f"(from {os.path.basename(fpath)})"
                    )
                    best_group_scores[group_id] = score
                    best_group_data[group_id] = data_map[group_id]

    # 3. Construct final result
    print("\nConstructing final ensemble CSV...")
    final_rows = []

    # Sort by Group ID (usually preferred by the problem, though not mandatory)
    # Assume Group ID is a numeric string like '003', '200'
    sorted_groups = sorted(best_group_data.keys(), key=lambda x: int(x))

    for gid in sorted_groups:
        group_df = best_group_data[gid]

        # Reassign item_id order (0..N)
        # Some files may have shuffled order, so reset based on row index
        for i, (_, row) in enumerate(group_df.iterrows()):
            final_rows.append({
                'id': f"{gid}_{i}",
                'x': f"s{row['x']}",
                'y': f"s{row['y']}",
                'deg': f"s{row['deg']}"
            })

    final_df = pd.DataFrame(final_rows)
    final_df = final_df[['id', 'x', 'y', 'deg']]
    final_df.to_csv(output_file, index=False)

    # 4. Compute final ensemble score
    final_score = get_total_score(best_group_scores)
    print(f"\nEnsemble Complete!")
    print(f"Saved to: {output_file}")
    print(f"Estimated Total Score: {final_score:.8f}")

if __name__ == '__main__':
    # --- Configuration ---
    # Option 2: manually specify file list (recommended)
    csv_files = [
        '/kaggle/input/santa2025-solutions-guided-refinement/submission.csv',        # your original file
        '/kaggle/input/santa25-solutions-optimization-visualization/submission.csv',        # file generated by the optimized code
        '/kaggle/input/the-boxes-shrunk/submission.csv',
        '/kaggle/input/why-not/submission.csv',
        '/kaggle/input/christmas-spirit/submission.csv',
        '/kaggle/input/santa-claude/submission.csv',
        # 'other_high_score.csv', # other high-score sources if available
    ]

    ensemble_files(csv_files)

Loading: /kaggle/input/santa2025-solutions-guided-refinement/submission.csv
Loading: /kaggle/input/santa25-solutions-optimization-visualization/submission.csv
  [Group 001] Improved! 0.813173 -> 0.813173 (from submission.csv)
Loading: /kaggle/input/the-boxes-shrunk/submission.csv
  [Group 003] Improved! 1.142031 -> 1.142031 (from submission.csv)
  [Group 004] Improved! 1.290822 -> 1.290808 (from submission.csv)
  [Group 005] Improved! 1.443702 -> 1.443693 (from submission.csv)
  [Group 006] Improved! 1.548443 -> 1.548440 (from submission.csv)
  [Group 007] Improved! 1.673119 -> 1.673110 (from submission.csv)
  [Group 008] Improved! 1.766811 -> 1.766781 (from submission.csv)
  [Group 009] Improved! 1.867531 -> 1.867324 (from submission.csv)
  [Group 010] Improved! 1.940714 -> 1.940713 (from submission.csv)
  [Group 011] Improved! 2.034227 -> 2.033028 (from submission.csv)
  [Group 012] Improved! 2.115066 -> 2.114978 (from submission.csv)
  [Group 013] Improved! 2.200253 -> 2.200065 (fro

# SA

In [2]:
import pandas as pd
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
from shapely.strtree import STRtree
import time
import multiprocessing
import math
import random
import os
from collections import defaultdict
import argparse

# --- Global configuration ---
getcontext().prec = 50
scale_factor = Decimal('1e18')

# --- Core class definition ---
class ChristmasTree:
    def __init__(self, center_x='0', center_y='0', angle='0', item_id=None):
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)
        self.item_id = item_id
        self.polygon = self._create_polygon()

    def _create_polygon(self):
        trunk_w = Decimal('0.15'); trunk_h = Decimal('0.2')
        base_w = Decimal('0.7'); base_y = Decimal('0.0')
        mid_w = Decimal('0.4'); tier_2_y = Decimal('0.25')
        top_w = Decimal('0.25'); tier_1_y = Decimal('0.5')
        tip_y = Decimal('0.8'); trunk_bottom_y = -trunk_h

        initial_polygon = Polygon([
            (Decimal('0.0') * scale_factor, tip_y * scale_factor),
            (top_w / Decimal('2') * scale_factor, tier_1_y * scale_factor),
            (top_w / Decimal('4') * scale_factor, tier_1_y * scale_factor),
            (mid_w / Decimal('2') * scale_factor, tier_2_y * scale_factor),
            (mid_w / Decimal('4') * scale_factor, tier_2_y * scale_factor),
            (base_w / Decimal('2') * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal('2') * scale_factor, base_y * scale_factor),
            (trunk_w / Decimal('2') * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal('2')) * scale_factor, trunk_bottom_y * scale_factor),
            (-(trunk_w / Decimal('2')) * scale_factor, base_y * scale_factor),
            (-(base_w / Decimal('2')) * scale_factor, base_y * scale_factor),
            (-(mid_w / Decimal('4')) * scale_factor, tier_2_y * scale_factor),
            (-(mid_w / Decimal('2')) * scale_factor, tier_2_y * scale_factor),
            (-(top_w / Decimal('4')) * scale_factor, tier_1_y * scale_factor),
            (-(top_w / Decimal('2')) * scale_factor, tier_1_y * scale_factor),
        ])
        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        return affinity.translate(
            rotated,
            xoff=float(self.center_x * scale_factor),
            yoff=float(self.center_y * scale_factor)
        )

    def clone(self) -> "ChristmasTree":
        new_tree = ChristmasTree.__new__(ChristmasTree)
        new_tree.center_x = self.center_x
        new_tree.center_y = self.center_y
        new_tree.angle = self.angle
        new_tree.item_id = self.item_id
        new_tree.polygon = self.polygon
        return new_tree


def splitmix64(x: int) -> int:
    """Deterministic 64-bit mixer (same spirit as C++ SplitMix64)."""
    x &= 0xFFFFFFFFFFFFFFFF
    x = (x + 0x9E3779B97F4A7C15) & 0xFFFFFFFFFFFFFFFF
    z = x
    z = (z ^ (z >> 30)) * 0xBF58476D1CE4E5B9 & 0xFFFFFFFFFFFFFFFF
    z = (z ^ (z >> 27)) * 0x94D049BB133111EB & 0xFFFFFFFFFFFFFFFF
    z = z ^ (z >> 31)
    return z & 0xFFFFFFFFFFFFFFFF


def parse_int_range(s: str):
    """Parse '40-80' (inclusive) or a single int like '12'. Returns (min,max) or None."""
    if s is None:
        return None
    s = str(s).strip()
    if not s:
        return None
    if '-' in s:
        a, b = s.split('-', 1)
        a = int(a.strip())
        b = int(b.strip())
        if a > b:
            a, b = b, a
        return a, b
    v = int(s)
    return v, v

# --- Fast helper functions ---


def get_tree_list_side_length_fast(polygons) -> float:
    """Fast side-length computation (float precision)."""
    if not polygons:
        return 0.0
    minx, miny, maxx, maxy = polygons[0].bounds
    for p in polygons[1:]:
        b = p.bounds
        if b[0] < minx: minx = b[0]
        if b[1] < miny: miny = b[1]
        if b[2] > maxx: maxx = b[2]
        if b[3] > maxy: maxy = b[3]
    return max(maxx - minx, maxy - miny) / float(scale_factor)


def validate_no_overlaps(polygons):
    """Final safety check: use STRtree to detect physical overlap (avoid expensive intersection().area)."""
    if not polygons:
        return True

    strtree = STRtree(polygons)

    for i, poly in enumerate(polygons):
        candidates = strtree.query(poly)

        for cand in candidates:
            # Shapely 1.8: query returns geometries; Shapely 2.x: often returns indices (depends on construction)
            if hasattr(cand, "geom_type"):
                other = cand
                if other is poly:
                    continue
            else:
                j = int(cand)
                if j == i:
                    continue
                other = polygons[j]

            # touches (edge/point contact) is allowed; any non-disjoint and non-touching is treated as area overlap
            if (not poly.disjoint(other)) and (not poly.touches(other)):
                return False

    return True


def parse_csv(csv_path):
    print(f'Loading csv: {csv_path}')
    result = pd.read_csv(csv_path)
    for col in ['x', 'y', 'deg']:
        if result[col].dtype == object:
            result[col] = result[col].astype(str).str.strip('s')

    # id is usually like "<group>_<item>"; keep item_id so we can preserve IDs on save.
    result[['group_id', 'item_id']] = result['id'].str.split('_', n=1, expand=True)

    dict_of_tree_list = {}
    for group_id, group_data in result.groupby('group_id'):
        # iterrows -> itertuples (faster)
        tree_list = [
            ChristmasTree(center_x=str(row.x), center_y=str(row.y), angle=str(row.deg), item_id=str(row.item_id))
            for row in group_data.itertuples(index=False)
        ]
        dict_of_tree_list[group_id] = tree_list
    return dict_of_tree_list


def save_dict_to_csv(dict_of_tree_list, output_path):
    print(f"Saving solution to {output_path}...")
    data = []
    sorted_keys = sorted(dict_of_tree_list.keys(), key=lambda x: int(x))
    for group_id in sorted_keys:
        trees = dict_of_tree_list[group_id]
        for i, tree in enumerate(trees):
            item_id = tree.item_id if tree.item_id is not None else str(i)
            data.append({
                'id': f"{group_id}_{item_id}",
                'x': f"s{tree.center_x}",
                'y': f"s{tree.center_y}",
                'deg': f"s{tree.angle}",
            })
    df = pd.DataFrame(data)[['id', 'x', 'y', 'deg']]
    df.to_csv(output_path, index=False)
    print("Save complete.")


# --- Simulated Annealing worker ---


def run_simulated_annealing(args):
    group_id, initial_trees, max_iterations, t_start, t_end, base_seed = args
    n_trees = len(initial_trees)

    gid_int = int(group_id)
    task_seed = splitmix64((int(base_seed) ^ (gid_int * 0x9E3779B97F4A7C15)) & 0xFFFFFFFFFFFFFFFF)
    rng = random.Random(task_seed)

    # Decide by N size
    is_small_n = n_trees <= 50

    if is_small_n:
        effective_max_iter = max_iterations * 3
        effective_t_start = t_start * 2.0
        gravity_weight = 1e-4
    else:
        effective_max_iter = max_iterations
        effective_t_start = t_start
        gravity_weight = 1e-6

    # Initialize state
    state = []
    for t in initial_trees:
        cx_float = float(t.center_x) * float(scale_factor)
        cy_float = float(t.center_y) * float(scale_factor)
        state.append({
            'poly': t.polygon,
            'cx': cx_float,
            'cy': cy_float,
            'angle': float(t.angle),
        })

    current_polys = [s['poly'] for s in state]
    current_bounds = [p.bounds for p in current_polys]

    scale_f = float(scale_factor)
    inv_scale_f = 1.0 / scale_f
    inv_scale_f2 = 1.0 / (scale_f * scale_f)

    def _envelope_from_bounds(bounds_list):
        if not bounds_list:
            return (0.0, 0.0, 0.0, 0.0)
        minx, miny, maxx, maxy = bounds_list[0]
        for b in bounds_list[1:]:
            if b[0] < minx: minx = b[0]
            if b[1] < miny: miny = b[1]
            if b[2] > maxx: maxx = b[2]
            if b[3] > maxy: maxy = b[3]
        return (minx, miny, maxx, maxy)

    def _envelope_from_bounds_replace(bounds_list, replace_i: int, replace_bounds):
        """Compute the envelope after replacing bounds_list[replace_i] without mutating the list."""
        if not bounds_list:
            return (0.0, 0.0, 0.0, 0.0)
        b0 = replace_bounds if replace_i == 0 else bounds_list[0]
        minx, miny, maxx, maxy = b0
        for i, b in enumerate(bounds_list[1:], start=1):
            if i == replace_i:
                b = replace_bounds
            if b[0] < minx: minx = b[0]
            if b[1] < miny: miny = b[1]
            if b[2] > maxx: maxx = b[2]
            if b[3] > maxy: maxy = b[3]
        return (minx, miny, maxx, maxy)

    def _side_len_from_env(env):
        minx, miny, maxx, maxy = env
        return max(maxx - minx, maxy - miny) * inv_scale_f

    # Initialize envelope & dist_sum (maintained incrementally later)
    env = _envelope_from_bounds(current_bounds)
    dist_sum = 0.0
    for s in state:
        dist_sum += s['cx'] * s['cx'] + s['cy'] * s['cy']

    def energy_from(env_local, dist_sum_local):
        side_len = _side_len_from_env(env_local)
        normalized_dist = (dist_sum_local * inv_scale_f2) / max(1, n_trees)
        return side_len + gravity_weight * normalized_dist, side_len

    current_energy, current_side_len = energy_from(env, dist_sum)

    best_state_params = [{'cx': s['cx'], 'cy': s['cy'], 'angle': s['angle']} for s in state]
    best_real_score = current_side_len

    T = effective_t_start
    cooling_rate = math.pow(t_end / effective_t_start, 1.0 / effective_max_iter)

    for i in range(effective_max_iter):
        progress = i / effective_max_iter

        if is_small_n:
            move_scale = max(0.005, 3.0 * (1 - progress))
            rotate_scale = max(0.001, 5.0 * (1 - progress))
        else:
            move_scale = max(0.001, 1.0 * (T / effective_t_start))
            rotate_scale = max(0.002, 5.0 * (T / effective_t_start))

        idx = rng.randint(0, n_trees - 1)
        target = state[idx]

        orig_poly = target['poly']
        orig_bounds = current_bounds[idx]
        orig_cx, orig_cy, orig_angle = target['cx'], target['cy'], target['angle']

        dx = (rng.random() - 0.5) * scale_f * 0.1 * move_scale
        dy = (rng.random() - 0.5) * scale_f * 0.1 * move_scale
        d_angle = (rng.random() - 0.5) * rotate_scale

        rotated_poly = affinity.rotate(orig_poly, d_angle, origin=(orig_cx, orig_cy))
        new_poly = affinity.translate(rotated_poly, xoff=dx, yoff=dy)
        new_bounds = new_poly.bounds
        minx, miny, maxx, maxy = new_bounds

        new_cx = orig_cx + dx
        new_cy = orig_cy + dy
        new_angle = orig_angle + d_angle

        # --- Collision detection: fall back to full scan with bbox pruning ---
        collision = False
        for k in range(n_trees):
            if k == idx:
                continue
            ox1, oy1, ox2, oy2 = current_bounds[k]
            if maxx < ox1 or minx > ox2 or maxy < oy1 or miny > oy2:
                continue
            other = current_polys[k]
            # touches (edge/point contact) is allowed; any non-disjoint and non-touching is treated as overlap
            if (not new_poly.disjoint(other)) and (not new_poly.touches(other)):
                collision = True
                break

        if collision:
            T *= cooling_rate
            continue

        # Incremental update for dist_sum
        old_d = orig_cx * orig_cx + orig_cy * orig_cy
        new_d = new_cx * new_cx + new_cy * new_cy
        cand_dist_sum = dist_sum - old_d + new_d

        # Incremental update for envelope: only rescan if it "breaks" current extrema
        env_minx, env_miny, env_maxx, env_maxy = env
        need_recompute = (
            (orig_bounds[0] == env_minx and new_bounds[0] > env_minx) or
            (orig_bounds[1] == env_miny and new_bounds[1] > env_miny) or
            (orig_bounds[2] == env_maxx and new_bounds[2] < env_maxx) or
            (orig_bounds[3] == env_maxy and new_bounds[3] < env_maxy)
        )
        if need_recompute:
            cand_env = _envelope_from_bounds_replace(current_bounds, idx, new_bounds)
        else:
            cand_env = (
                min(env_minx, new_bounds[0]),
                min(env_miny, new_bounds[1]),
                max(env_maxx, new_bounds[2]),
                max(env_maxy, new_bounds[3]),
            )

        new_energy, new_real_score = energy_from(cand_env, cand_dist_sum)
        delta = new_energy - current_energy

        accept = False
        if delta < 0:
            accept = True
        else:
            if T > 1e-10:
                prob = math.exp(-delta * 1000 / T)
                accept = rng.random() < prob

        if accept:
            current_polys[idx] = new_poly
            current_bounds[idx] = new_bounds
            target['poly'] = new_poly
            target['cx'] = new_cx
            target['cy'] = new_cy
            target['angle'] = new_angle

            current_energy = new_energy
            env = cand_env
            dist_sum = cand_dist_sum

            if new_real_score < best_real_score:
                best_real_score = new_real_score
                for k in range(n_trees):
                    best_state_params[k]['cx'] = state[k]['cx']
                    best_state_params[k]['cy'] = state[k]['cy']
                    best_state_params[k]['angle'] = state[k]['angle']

        T *= cooling_rate

    final_trees = []
    final_polys_check = []
    for p in best_state_params:
        cx_dec = Decimal(p['cx']) / scale_factor
        cy_dec = Decimal(p['cy']) / scale_factor
        angle_dec = Decimal(p['angle'])
        new_t = ChristmasTree(str(cx_dec), str(cy_dec), str(angle_dec))
        final_trees.append(new_t)
        final_polys_check.append(new_t.polygon)

    if not validate_no_overlaps(final_polys_check):
        orig_score = get_tree_list_side_length_fast([t.polygon for t in initial_trees])
        return group_id, initial_trees, orig_score

    return group_id, final_trees, best_real_score


# --- Main logic ---
def main():
    parser = argparse.ArgumentParser(description="Santa-2025 SA optimizer (Python/Shapely).")
    parser.add_argument("--input", default="/kaggle/working/submission_.csv", help="Input CSV path")
    parser.add_argument("--output", default="/kaggle/working/submission.csv", help="Output CSV path")
    parser.add_argument("--iter", type=int, default=1000000, help="Base iterations per group")
    parser.add_argument("--tstart", type=float, default=10.0, help="Start temperature")
    parser.add_argument("--tend", type=float, default=0.01, help="End temperature")
    parser.add_argument("--processes", default="auto", help="Process count or 'auto'")
    parser.add_argument("--seed", type=int, default=42, help="Deterministic base seed")
    parser.add_argument("--range", default=None, help="Only optimize groups in inclusive range a-b")
    parser.add_argument("--gid_min", type=int, default=None, help="Only optimize groups >= gid_min")
    parser.add_argument("--gid_max", type=int, default=None, help="Only optimize groups <= gid_max")
    parser.add_argument("--time_limit_sec", type=int, default=11.5 * 3600, help="Wall time limit")
    parser.add_argument("--save_every", type=int, default=20, help="Checkpoint frequency by finished groups")
    args,_ = parser.parse_known_args()

    INPUT_CSV = args.input
    OUTPUT_CSV = args.output

    try:
        dict_of_tree_list = parse_csv(INPUT_CSV)
    except FileNotFoundError:
        print(f"Error: Could not find {INPUT_CSV}.")
        return

    all_groups_sorted = sorted(dict_of_tree_list.keys(), key=lambda x: int(x), reverse=True)

    gid_min = args.gid_min
    gid_max = args.gid_max
    r = parse_int_range(args.range)
    if r is not None:
        gid_min, gid_max = r

    if gid_min is None:
        gid_min = -10**18
    if gid_max is None:
        gid_max = 10**18

    groups_to_optimize = [gid for gid in all_groups_sorted if gid_min <= int(gid) <= gid_max]

    MAX_ITER = int(args.iter)
    T_START = float(args.tstart)
    T_END = float(args.tend)

    KAGGLE_TIME_LIMIT_SEC = int(args.time_limit_sec)
    SAVE_EVERY_N_GROUPS = int(args.save_every)

    tasks = []
    for gid in groups_to_optimize:
        tasks.append((gid, dict_of_tree_list[gid], MAX_ITER, T_START, T_END, args.seed))

    if str(args.processes).lower() == "auto":
        num_processes = multiprocessing.cpu_count()
    else:
        num_processes = max(1, int(args.processes))

    # Don't spawn more workers than tasks.
    num_processes = min(num_processes, max(1, len(tasks)))

    print(f"Starting SA on {len(tasks)}/{len(all_groups_sorted)} groups using {num_processes} processes...")
    if gid_min != -10**18 or gid_max != 10**18:
        print(f"Group filter: {gid_min}-{gid_max} (inclusive)")
    print(f"Seed(base): {args.seed}")
    print(f"Time Limit: {KAGGLE_TIME_LIMIT_SEC / 3600:.2f} hours")
    print("Press Ctrl+C to stop early and save progress.")

    start_time = time.time()
    improved_count = 0
    total_tasks = len(tasks)
    finished_tasks = 0

    pool = multiprocessing.Pool(processes=num_processes)

    try:
        results_iter = pool.imap_unordered(run_simulated_annealing, tasks, chunksize=1)

        for result in results_iter:
            group_id, optimized_trees, score = result
            finished_tasks += 1

            orig_polys = [t.polygon for t in dict_of_tree_list[group_id]]
            orig_score = get_tree_list_side_length_fast(orig_polys)

            status_msg = ""
            if score < orig_score:
                diff = orig_score - score
                if diff > 1e-12:
                    status_msg = f" -> Improved! (-{diff:.6f})"
                    dict_of_tree_list[group_id] = optimized_trees
                    improved_count += 1

            elapsed_time = time.time() - start_time
            if elapsed_time > KAGGLE_TIME_LIMIT_SEC:
                print(
                    f"\n[WARNING] Time limit approach ({elapsed_time / 3600:.2f}h). "
                    "Stopping early to save data safely."
                )
                pool.terminate()
                break

            if finished_tasks % SAVE_EVERY_N_GROUPS == 0:
                print(
                    f"   >>> Auto-saving checkpoint at "
                    f"{finished_tasks}/{total_tasks}..."
                )
                save_dict_to_csv(dict_of_tree_list, OUTPUT_CSV)

            print(
                f"[{finished_tasks}/{total_tasks}] "
                f"G:{group_id} {orig_score:.5f}->{score:.5f} {status_msg}"
            )

        pool.close()
        pool.join()
        print(f"\nOptimization finished normally in {time.time() - start_time:.2f}s")

    except KeyboardInterrupt:
        print("\n\n!!! Caught Ctrl+C (KeyboardInterrupt) !!!")
        print("Terminating workers and saving current progress...")
        pool.terminate()
        pool.join()
    except Exception as e:
        print(f"\nAn error occurred: {e}")
        pool.terminate()
        pool.join()
    finally:
        print(f"Final Save. Total Improved: {improved_count}")
        save_dict_to_csv(dict_of_tree_list, OUTPUT_CSV)


if __name__ == '__main__':
    multiprocessing.freeze_support()
    main()


Loading csv: /kaggle/working/submission_.csv
Starting SA on 200/200 groups using 4 processes...
Seed(base): 42
Time Limit: 11.50 hours
Press Ctrl+C to stop early and save progress.
[1/200] G:197 8.19890->8.19890 
[2/200] G:200 8.23334->8.23334 
[3/200] G:199 8.22827->8.22827 
[4/200] G:198 8.22300->8.22300 
[5/200] G:193 8.10655->8.10655 
[6/200] G:195 8.14433->8.14433 
[7/200] G:194 8.12760->8.12760 
[8/200] G:196 8.16646->8.16646 
[9/200] G:192 8.02685->8.02685 
[10/200] G:190 8.01811->8.01811 
[11/200] G:191 8.02228->8.02228 
[12/200] G:189 8.01008->8.01008 
[13/200] G:188 7.99671->7.99671 
[14/200] G:187 7.98308->7.98308 
[15/200] G:186 7.97134->7.97134 
[16/200] G:185 7.94711->7.94711 
[17/200] G:184 7.93361->7.93361 
[18/200] G:183 7.90160->7.90160 
[19/200] G:182 7.86449->7.86449 
   >>> Auto-saving checkpoint at 20/200...
Saving solution to /kaggle/working/submission.csv...
Save complete.
[20/200] G:181 7.86181->7.86181 
[21/200] G:180 7.86176->7.86176 
[22/200] G:179 7.86006->

# greedy_backtracking

In [3]:
%%writefile GB.py
import pandas as pd
from decimal import Decimal, getcontext
from shapely import affinity
from shapely.geometry import Polygon
import time, random
import multiprocessing as mp

# --- Global settings ---
getcontext().prec = 50
scale_factor = Decimal('1e18')
scale_factor_float = 1e18

# ====== Hyperparameters: more aggressive = larger ======
PASSES = 6                  # Multiple synchronized passes (parallel-friendly)
EPS_IMPROVE = 1e-12
BOUND_EPS = 1.0

DEPTH = 10                   # Can be larger, but should be paired with BEAM/MAX_STATES
BEAM = 10
MAX_STATES = 4000

RAND_TRIES = 8
RAND_K = 50
RANDOM_SEED = 42

# ====== Parallel settings ======
PROCESSES = max(1, mp.cpu_count() - 2)  # Leave some CPU for the OS
CHUNKSIZE = 8                           # map chunk size; tune per machine


# --- Core class definition ---
class ChristmasTree:
    def __init__(self, center_x='0', center_y='0', angle='0'):
        self.center_x = Decimal(center_x)
        self.center_y = Decimal(center_y)
        self.angle = Decimal(angle)
        self.polygon = self._create_polygon()
        self.bounds = self.polygon.bounds  # (minx, miny, maxx, maxy)

    def _create_polygon(self):
        trunk_w = Decimal('0.15'); trunk_h = Decimal('0.2')
        base_w = Decimal('0.7'); base_y = Decimal('0.0')
        mid_w  = Decimal('0.4'); tier_2_y = Decimal('0.25')
        top_w  = Decimal('0.25'); tier_1_y = Decimal('0.5')
        tip_y  = Decimal('0.8'); trunk_bottom_y = -trunk_h

        coords = [
            (Decimal('0.0'), tip_y),
            (top_w / 2, tier_1_y), (top_w / 4, tier_1_y),
            (mid_w / 2, tier_2_y), (mid_w / 4, tier_2_y),
            (base_w / 2, base_y), (trunk_w / 2, base_y),
            (trunk_w / 2, trunk_bottom_y), (-(trunk_w / 2), trunk_bottom_y),
            (-(trunk_w / 2), base_y), (-(base_w / 2), base_y),
            (-(mid_w / 4), tier_2_y), (-(mid_w / 2), tier_2_y),
            (-(top_w / 4), tier_1_y), (-(top_w / 2), tier_1_y),
        ]

        scaled_coords = [(float(x) * scale_factor_float, float(y) * scale_factor_float) for x, y in coords]
        initial_polygon = Polygon(scaled_coords)

        rotated = affinity.rotate(initial_polygon, float(self.angle), origin=(0, 0))
        return affinity.translate(
            rotated,
            xoff=float(self.center_x) * scale_factor_float,
            yoff=float(self.center_y) * scale_factor_float
        )

    def clone(self) -> "ChristmasTree":
        new_tree = ChristmasTree.__new__(ChristmasTree)
        new_tree.center_x = self.center_x
        new_tree.center_y = self.center_y
        new_tree.angle = self.angle
        new_tree.polygon = self.polygon
        new_tree.bounds = self.bounds
        return new_tree


# --- Bounds utilities ---
def get_bounds_side(bounds_list):
    if not bounds_list:
        return 0.0
    min_x = min(b[0] for b in bounds_list)
    min_y = min(b[1] for b in bounds_list)
    max_x = max(b[2] for b in bounds_list)
    max_y = max(b[3] for b in bounds_list)
    return max(max_x - min_x, max_y - min_y) / scale_factor_float

def compute_touching_candidates(bounds_list, eps=BOUND_EPS):
    n = len(bounds_list)
    if n == 0:
        return []
    min_x = min(b[0] for b in bounds_list)
    min_y = min(b[1] for b in bounds_list)
    max_x = max(b[2] for b in bounds_list)
    max_y = max(b[3] for b in bounds_list)
    cand = []
    for i, b in enumerate(bounds_list):
        if (abs(b[0] - min_x) < eps or abs(b[1] - min_y) < eps or
            abs(b[2] - max_x) < eps or abs(b[3] - max_y) < eps):
            cand.append(i)
    if not cand:
        cand = list(range(n))
    return cand


def choose_removal_beam_lookahead(bounds_list, depth, beam, max_states, rand_tries, rand_k, seed):
    """
    depth-step lookahead + beam search + random perturbations.
    Returns (best_first_idx, side_after_first_remove).
    """
    rng = random.Random(seed)
    n0 = len(bounds_list)
    if n0 <= 1:
        return None, 0.0

    def run_once(shuffle=True, limit_k=rand_k):
        base_cands = compute_touching_candidates(bounds_list)
        if shuffle:
            rng.shuffle(base_cands)
        if limit_k and len(base_cands) > limit_k:
            base_cands = base_cands[:limit_k]

        # First layer
        first_layer = []
        for idx in base_cands:
            reduced = bounds_list[:idx] + bounds_list[idx+1:]
            s1 = get_bounds_side(reduced)
            # (score_now, reduced_bounds, first_idx, first_s1)
            first_layer.append((s1, reduced, idx, s1))

        if not first_layer:
            return None, float("inf")

        first_layer.sort(key=lambda x: x[0])
        frontier = first_layer[:min(beam, len(first_layer))]

        # best_key: (future_best, first_s1)
        best_key = (frontier[0][0], frontier[0][3])
        best_first = frontier[0][2]
        best_s1 = frontier[0][3]

        states_used = len(frontier)

        # Expand to depth
        for _d in range(2, max(2, depth + 1)):
            new_frontier = []
            for score_now, bds, first_idx, first_s1 in frontier:
                if len(bds) <= 1:
                    key = (0.0, first_s1)
                    if key < best_key:
                        best_key, best_first, best_s1 = key, first_idx, first_s1
                    continue

                cands = compute_touching_candidates(bds)
                if shuffle:
                    rng.shuffle(cands)
                if limit_k and len(cands) > limit_k:
                    cands = cands[:limit_k]

                for j in cands:
                    nb = bds[:j] + bds[j+1:]
                    s = get_bounds_side(nb)
                    new_frontier.append((s, nb, first_idx, first_s1))
                    states_used += 1
                    if states_used >= max_states:
                        break
                if states_used >= max_states:
                    break

            if not new_frontier:
                break

            new_frontier.sort(key=lambda x: x[0])
            frontier = new_frontier[:min(beam, len(new_frontier))]

            cur_best = frontier[0]
            key = (cur_best[0], cur_best[3])
            if key < best_key:
                best_key, best_first, best_s1 = key, cur_best[2], cur_best[3]

            if states_used >= max_states:
                break

        return best_first, best_s1

    # First, do one "deterministic" run
    best_first, best_s1 = run_once(shuffle=False, limit_k=None)
    best_key = (best_s1, best_s1)

    # Multiple randomized runs: keep the best
    for _ in range(rand_tries):
        first, s1 = run_once(shuffle=True, limit_k=rand_k)
        if first is None:
            continue
        key = (s1, s1)
        if key < best_key:
            best_key = key
            best_first = first
            best_s1 = s1

    return best_first, best_s1


# --- Parallel worker: process one N and propose an improvement for N-1 ---
def worker_propose(args):
    """
    Input:
      (N, bounds_list, prev_best, depth, beam, max_states, rand_tries, rand_k, base_seed)
    Output:
      (target_gid, source_gid, remove_idx, new_side) or None
    """
    (N, bounds_list, prev_best, depth, beam, max_states, rand_tries, rand_k, base_seed) = args

    if bounds_list is None or len(bounds_list) <= 1:
        return None

    target_gid = f"{N-1:03d}"
    source_gid = f"{N:03d}"

    seed = (base_seed * 1000003) ^ (N * 9176) ^ (len(bounds_list) * 131)
    best_idx, best_s1 = choose_removal_beam_lookahead(
        bounds_list, depth, beam, max_states, rand_tries, rand_k, seed
    )

    if best_idx is None:
        return None

    if best_s1 < prev_best - EPS_IMPROVE:
        return (target_gid, source_gid, best_idx, best_s1)

    return None


# --- IO ---
def parse_csv(csv_path):
    print(f'Loading csv: {csv_path}')
    result = pd.read_csv(csv_path)

    for col in ['x', 'y', 'deg']:
        if result[col].dtype == object:
            result[col] = result[col].astype(str).str.strip().str.lstrip('s')

    result[['group_id', 'item_id']] = result['id'].astype(str).str.split('_', n=2, expand=True)

    dict_of_tree_list = {}
    dict_of_side_length = {}

    for group_id, group_data in result.groupby('group_id'):
        trees = []
        for _, row in group_data.iterrows():
            trees.append(ChristmasTree(center_x=row['x'], center_y=row['y'], angle=row['deg']))
        gid = f"{int(group_id):03d}"
        dict_of_tree_list[gid] = trees
        dict_of_side_length[gid] = get_bounds_side([t.bounds for t in trees])

    return dict_of_tree_list, dict_of_side_length


def save_dict_to_csv(dict_of_tree_list, output_path):
    print(f"Saving to {output_path}...")
    data = []
    sorted_keys = sorted(dict_of_tree_list.keys(), key=lambda x: int(x))
    for group_id in sorted_keys:
        trees = dict_of_tree_list[group_id]
        for i, tree in enumerate(trees):
            data.append({
                'id': f"{group_id}_{i}",
                'x': f"s{tree.center_x}",
                'y': f"s{tree.center_y}",
                'deg': f"s{tree.angle}"
            })
    pd.DataFrame(data)[['id', 'x', 'y', 'deg']].to_csv(output_path, index=False)
    print("Save complete.")


def main():
    INPUT_CSV = '/kaggle/working/submission.csv'
    OUTPUT_CSV = '/kaggle/working/submission.csv'

    try:
        dict_of_tree_list, dict_of_side_length = parse_csv(INPUT_CSV)
    except FileNotFoundError:
        print(f"Error: Not found {INPUT_CSV}")
        return

    start_time = time.time()
    print(
        f"MP HARD optimization: PROCESSES={PROCESSES}, PASSES={PASSES}, "
        f"DEPTH={DEPTH}, BEAM={BEAM}, MAX_STATES={MAX_STATES}, RAND_TRIES={RAND_TRIES}, RAND_K={RAND_K}"
    )

    # Windows compatibility: use spawn
    ctx = mp.get_context("spawn")

    changed_total = 0

    with ctx.Pool(processes=PROCESSES) as pool:
        for pass_id in range(1, PASSES + 1):
            print(f"\n=== PASS {pass_id}/{PASSES} ===")
            # Snapshot: all proposals in this pass are based on the same frozen data (parallel-safe)
            snap_tree_list = {k: v for k, v in dict_of_tree_list.items()}
            snap_side = {k: v for k, v in dict_of_side_length.items()}

            tasks = []
            base_seed = RANDOM_SEED + pass_id * 10007

            # Propose from N=200..3 to improve N-1
            for N in range(200, 2, -1):
                gidN = f"{N:03d}"
                gidPrev = f"{N-1:03d}"
                if gidN not in snap_tree_list or gidPrev not in snap_side:
                    continue
                bounds_list = [t.bounds for t in snap_tree_list[gidN]]
                prev_best = snap_side[gidPrev]
                tasks.append((N, bounds_list, prev_best, DEPTH, BEAM, MAX_STATES, RAND_TRIES, RAND_K, base_seed))

            # Compute proposals in parallel
            proposals = pool.map(worker_propose, tasks, chunksize=CHUNKSIZE)

            # For each target_gid, keep the best proposal (multiple N may target the same N-1)
            best_for_target = {}  # target_gid -> (new_side, source_gid, remove_idx)
            for p in proposals:
                if p is None:
                    continue
                target_gid, source_gid, remove_idx, new_side = p
                cur = best_for_target.get(target_gid)
                if cur is None or new_side < cur[0]:
                    best_for_target[target_gid] = (new_side, source_gid, remove_idx)

            # Apply improvements (single unified write-back)
            changed_this_pass = 0
            for target_gid, (new_side, source_gid, remove_idx) in best_for_target.items():
                old_side = dict_of_side_length.get(target_gid, float("inf"))
                if new_side < old_side - EPS_IMPROVE:
                    # Remove remove_idx from the snapshot solution of source_gid to form the new solution for target_gid
                    src_list = snap_tree_list[source_gid]
                    new_list = src_list[:remove_idx] + src_list[remove_idx+1:]
                    dict_of_tree_list[target_gid] = new_list
                    dict_of_side_length[target_gid] = new_side
                    print(f"[Group {target_gid}] Improved! {old_side:.6f} -> {new_side:.6f} (from {source_gid}, rm={remove_idx})")
                    changed_this_pass += 1
                    changed_total += 1

            print(f"PASS {pass_id} changes: {changed_this_pass}")
            if changed_this_pass == 0:
                print("No changes -> early stop.")
                break

    print(f"\nTotal changes: {changed_total}")
    print(f"Total time: {time.time() - start_time:.2f}s")
    save_dict_to_csv(dict_of_tree_list, OUTPUT_CSV)


if __name__ == '__main__':
    main()

Writing GB.py


In [4]:
!python GB.py

Loading csv: /kaggle/working/submission.csv
MP HARD optimization: PROCESSES=2, PASSES=6, DEPTH=10, BEAM=10, MAX_STATES=4000, RAND_TRIES=8, RAND_K=50

=== PASS 1/6 ===
PASS 1 changes: 0
No changes -> early stop.

Total changes: 0
Total time: 10.40s
Saving to /kaggle/working/submission.csv...
Save complete.
